# 00 - Environment and claim boundary

## Objective

Identify the installed OpenDSS and CEPT runtimes before doing a study. Learn that a green public workflow receipt is narrower than project or field validation.

## Source, assumptions, and units

The source is the installed CEPT public distribution and its bundled capability contract. This lesson does not invent a circuit or a measurement. Later lessons use volts as per unit (`pu`), power as kW/kvar, current as A, and distance as km where those quantities are returned.

The public runtime is OpenDSS-first. PowerFactory and dynamics are outside this notebook boundary. The examples are demonstrator workflows, not project validation.

## Prediction

Before running the CLI, predict that the public engine will be OpenDSS and that the four advertised study types will be marked `candidate`.

## Action

The cells below call the installed noun+verb CLI. Output is streamed from the subprocess into this notebook; a nonzero exit raises immediately.

## Verification

The assertions use the actual `capability show` and `system doctor` JSON returned by the installed runtime.

## Interpretation

`candidate` describes the public capability boundary. It is not a claim that a user's equipment, settings, measurements, or acceptance criteria have been validated.

## Exercise

Restart the kernel and run all cells. Then set `CEPT_WHEEL_URL` and `CEPT_WHEEL_SHA256` in the runtime environment to a caller-provided wheel and repeat the setup. Do not replace the hash with a guessed value.

## Runtime requirements

- Python 3.10 or newer.
- An existing installed `cept` command, or a caller-provided wheel URL and SHA-256 in `CEPT_WHEEL_URL` and `CEPT_WHEEL_SHA256`.
- The wheel supplies CEPT, OpenDSSDirect.py, and the native OpenDSS runtime.
- Jupyter/nbconvert is needed only to execute the notebook.

No released PyPI version is assumed.

In [ ]:
import hashlib
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import urllib.parse
import urllib.request
from pathlib import Path

DEFAULT_WHEEL_URL = 'https://github.com/sarutesri/cept-studio-edu/releases/download/v0.2.0-edu.1/cept_power_studio-0.2.0.dev0-py3-none-any.whl'
DEFAULT_WHEEL_SHA256 = 'c7e609a1d9cc85b322bfb615f0c796c7ea5c43815b197289eb555786964478bc'
configured_url = os.environ.get('CEPT_WHEEL_URL')
WHEEL_URL = (DEFAULT_WHEEL_URL if configured_url is None and importlib.util.find_spec('cept') is None else (configured_url or '')).strip()
WHEEL_SHA256 = os.environ.get('CEPT_WHEEL_SHA256', DEFAULT_WHEEL_SHA256 if WHEEL_URL == DEFAULT_WHEEL_URL else '').strip().lower()
if WHEEL_URL:
    if len(WHEEL_SHA256) != 64 or any(character not in '0123456789abcdef' for character in WHEEL_SHA256):
        raise ValueError('CEPT_WHEEL_SHA256 must be the caller-provided 64-character SHA-256')
    wheel_path = Path.cwd() / Path(urllib.parse.urlparse(WHEEL_URL).path).name
    print(f'Downloading caller-provided wheel: {WHEEL_URL}')
    urllib.request.urlretrieve(WHEEL_URL, wheel_path)
    digest = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if digest != WHEEL_SHA256:
        raise ValueError(f'wheel hash mismatch: expected {WHEEL_SHA256}, got {digest}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)], check=True)
else:
    print('CEPT_WHEEL_URL not supplied; using the existing installed environment.')

CLI_BIN = Path(sys.executable).parent / ('cept.exe' if os.name == 'nt' else 'cept')
if not CLI_BIN.is_file():
    raise RuntimeError('cept launcher not found next to Python; reinstall the pinned public wheel')
CLI = str(CLI_BIN)
print('CLI: cept --version')
print(subprocess.run([CLI, '--version'], capture_output=True, text=True, check=True).stdout.strip())
def run_cli(*arguments):
    display = 'cept ' + shlex.join([str(argument) for argument in arguments])
    command = [CLI, *[str(argument) for argument in arguments]]
    print('$ ' + display, flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd())
    lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    returncode = process.wait()
    output = ''.join(lines)
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ['cept', *[str(argument) for argument in arguments]], output=output)
    return json.loads(output) if output.strip().startswith('{') else output

def show_table(headers, rows):
    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join('---' for _ in headers) + ' |')
    for row in rows:
        print('| ' + ' | '.join(str(value) for value in row) + ' |')


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0


In [ ]:
capabilities = run_cli('capability', 'show')
doctor = run_cli('system', 'doctor')
show_table(
    ['study type', 'engine', 'status'],
    [
        (study, details['engine'], details['status'])
        for study, details in sorted(capabilities['capabilities'].items())
    ],
)
show_table(
    ['doctor field', 'actual value'],
    [('status', doctor['status']), ('engine', doctor['engine']), ('engine version', doctor['engine_version'])],
)
assert doctor['status'] == 'PASS'
assert doctor['engine'] == 'opendss'
assert set(capabilities['capabilities']) == {'fault', 'hosting_capacity', 'load_flow', 'unbalanced_load_flow'}
assert all(details['status'] == 'candidate' for details in capabilities['capabilities'].values())


$ cept capability show


{


  "capabilities": {


    "fault": {


      "engine": "opendss",


...

      "status": "not_supported_initially"


    }


  }


}


| study type | engine | status |
| --- | --- | --- |
| fault | opendss | candidate |
| hosting_capacity | opendss | candidate |
| load_flow | opendss | candidate |
| unbalanced_load_flow | opendss | candidate |
| doctor field | actual value |
| --- | --- |
| status | PASS |
| engine | opendss |
| engine version | DSS C-API Library version 0.14.5 revision 87d85c2622c8281b92255335bc7c09b11191b21d based on OpenDSS SVN 3723 [FPC 3.2.2] (64-bit build) MVMULT INCREMENTAL_Y CONTEXT_API PM 20240329033747; License Status: Open 
DSS-Python version: 0.15.7
OpenDSSDirect.py version: 0.9.4 |


The tables above are the installed CLI's actual response, not a recorded output. Keep the displayed version with later run artifacts when reporting a result. The public claim ceiling remains `WORKFLOW_VALIDATED`; no project-validation claim is created by this lesson.